<p align="center"><img src="https://raw.githubusercontent.com/yamenetoo/sup-score-growth/main/assets/logo.png" width="600"/></p>

# supgrow on Google Colab

This notebook installs **supgrow**, shows the test on one data set, and runs the experiments of the paper
*When Should a Neural Network Grow? A Sup-Score Test for Adding Hidden Units*.

**Before you start:** `Runtime → Change runtime type → GPU` (T4 is enough). Everything also runs on CPU, more slowly.

Results are written to `results/`; the last cell downloads them as a zip file.

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU: the code will run on CPU"
!git clone -q https://github.com/yamenetoo/sup-score-growth.git
%cd sup-score-growth
!pip install -q -e ".[cnn]"

In [ ]:
import torch, supgrow
print("supgrow", supgrow.__version__, "| torch", torch.__version__,
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. The test on one data set

Data from a two-unit tanh network (DGP-2 of the paper). `grow` starts from $K=0$ and adds a unit while the
bootstrap p-value of the sup-score test is at most $\alpha$.

In [ ]:
import numpy as np
from supgrow import grow, sup_score_test

rng = np.random.default_rng(0)
n = 500
X = rng.standard_normal((n, 2))
xt = np.c_[np.ones(n), X]
f = 1.5 * np.tanh(xt @ [0, 1, 0.5]) - 1.0 * np.tanh(xt @ [0.5, -0.5, 1.5])
y = f + 0.5 * rng.standard_normal(n)

res = grow(X, y, K_max=6, alpha=0.05, B=499, n_random=10, verbose=1)
print()
print(res.summary())

In [ ]:
# One test on its own: is a network with K = 1 unit sufficient?
out = sup_score_test(X, y, K=1)
print(f"T = {out['T']:.2f}, bootstrap p-value = {out['p_value']:.4f}, Bonferroni p-value = {out['p_bonferroni']:.4f}")

# Predictions of the selected network
X_new = rng.standard_normal((5, 2))
print(res.predict(X_new))

## 3. Experiments of the paper (E1–E7)

| Name | Paper | What it does |
|---|---|---|
| `size` | E1 | size and power of five tests |
| `local` | E2 | power against weak departures |
| `growth` | E3 | selection of the number of units, 4 designs, cost |
| `real` | E4 | four real data sets, 10-fold CV |
| `sens` | E5 | grid size / scale and variance estimator |
| `init` | E6 | one-step initialisation of the new unit |
| `deep` | E7 | layer-wise growth of a two-hidden-layer network |

First run the quick check (a few minutes), then the experiments you need. The full set of experiments takes
several hours on a T4; Colab may disconnect long sessions, so run them one at a time.

In [ ]:
# Quick check of every experiment (few replications)
!python -m supgrow.experiments all --quick --devices auto --verbose 0

In [ ]:
#@title Run selected experiments { display-mode: "form" }
size   = False  #@param {type:"boolean"}
local  = False  #@param {type:"boolean"}
growth = True   #@param {type:"boolean"}
real   = False  #@param {type:"boolean"}
sens   = False  #@param {type:"boolean"}
init   = False  #@param {type:"boolean"}
deep   = False  #@param {type:"boolean"}
batch  = 100    #@param {type:"integer"}
verbose = "0"   #@param ["0", "1", "2"]

chosen = [k for k, v in dict(size=size, local=local, growth=growth, real=real,
                             sens=sens, init=init, deep=deep).items() if v]
print("running:", chosen)
if chosen:
    !python -m supgrow.experiments {" ".join(chosen)} --devices auto --batch {batch} --verbose {verbose}

## 4. Convolutional networks (E8, exploratory)

Growth of a two-layer tanh CNN on MNIST and Fashion-MNIST: the proposed rule, a GradMax-type trigger, a
schedule, the selected widths retrained from scratch, and fixed widths trained for the same number of epochs.

In [ ]:
# Quick check (MNIST, 1 seed, small widths)
!python -m supgrow.cnn_growth --quick

In [ ]:
#@title Full CNN run { display-mode: "form" }
seeds = 5   #@param {type:"integer"}
data = "mnist fashion"  #@param ["mnist fashion", "mnist", "fashion"]
!python -m supgrow.cnn_growth --data {data} --seeds {seeds}

## 5. Results

In [ ]:
import glob, pandas as pd
from IPython.display import display, Markdown
for f in sorted(glob.glob("results/*.csv")):
    if f.endswith("_raw.csv"):
        continue
    display(Markdown(f"**{f}**"))
    display(pd.read_csv(f).round(4))

In [ ]:
# Download all results
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")